# M-generalization: does the student learn *abstract* units?

Each latent unit has `M = num_tuples` interchangeable surface **spellings** (tuples)
that all decode to the same id (disjoint supports). A model that formed the *abstract*
unit behaves the same regardless of which spelling appears; one that took a surface
shortcut does not. This notebook probes that on a trained model two ways:

1. **Overfitting to spellings** — fit a linear probe on positions whose target unit used
   a *train* subset of spellings, test on *held-out* spellings. A large seen–heldout gap
   = surface memorization. (logic: `src/analysis/spelling_generalization.py`)
2. **Prediction variability** — render the *same* latent sequence in several spellings and
   measure how much the student's predictions disagree. At unit boundaries the correct
   prediction is spelling-invariant (teacher divergence = 0), so the student's residual
   divergence there is pure surface leakage.

`DEMO = True` trains a small student inline so the notebook is self-contained; point it at
a real checkpoint for real results. Run from the repository root.

In [ ]:
%load_ext autoreload
%autoreload 2

import pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn.functional as F

from src.teachers import ChunkCode, LinearARTeacher, MultiLevelHierarchicalTeacher
from src.model.decoder import TransformerDecoder
from src.predictors import build_predictor
from src.data import ARDataset
from src.analysis.spelling_generalization import (
    spelling_generalization, summarize, render_spellings, cross_spelling_divergence,
)

# Validated categorical palette (see notebooks/head_phases.ipynb).
SERIES = ['#1b6ca8', '#d95f02', '#009E73']
torch.manual_seed(0)

## 1. A teacher with spellings, and a briefly-trained student

Two levels; the **bottom** level has `M = 3` spellings, so each surface bigram-of-units has
3 interchangeable realizations. The student is intentionally small and trained only a few
hundred steps — enough to show the *overfitting* regime. Swap this cell for checkpoint
loading to analyze a real run.

In [ ]:
base = LinearARTeacher.from_parameters(
    dim=16, span_lengths=[1, 1], window=2,
    spectrum={'rank': 16}, lag_spectrum={'law': 'geometric', 'decay': 1.5}, scale=8.0,
)
levels = [
    ChunkCode(in_dim=16, out_dim=10, size=2, num_tuples=1, chunk_seed=0),  # level 0 (M=1)
    ChunkCode(in_dim=10, out_dim=16, size=2, num_tuples=3, chunk_seed=1),  # level 1 (M=3)
]
teacher = MultiLevelHierarchicalTeacher(base, levels)
total, burn_in = teacher.total, teacher.burn_in

# Process-following training data.
predictor = build_predictor(teacher)
LEN = 24  # multiple of total; full sequence = burn_in + LEN
ds = ARDataset(predictor, window=teacher.window, dim=teacher.dim, number=2048,
               length=LEN, prefix_length=burn_in, unroll_sequences=False)
data = ds.data
seqlen = data.shape[1]

# Small student, teacher-forced next-token CE.
H = 64
student = TransformerDecoder(dim=teacher.dim, hidden_dim=H, num_heads=1, ff_hidden_dim=H,
    num_blocks=2, dropout=0.0, pe_type='absolute', pe_embedding_dim=H,
    pe_max_sequence_length=seqlen)
opt = torch.optim.Adam(student.parameters(), lr=1e-3)
N, BATCH, STEPS = data.shape[0], 256, 400
for step in range(STEPS):
    seq = data[torch.randint(0, N, (BATCH,))]
    logits = student(seq[:, :-1, :])[0]
    tgt = seq[:, 1:, :].argmax(-1)
    loss = F.cross_entropy(logits.reshape(-1, teacher.dim), tgt.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
print(f'total={total}  burn_in={burn_in}  seqlen={seqlen}  final train loss={loss.item():.3f}')

## 2. Overfitting to spellings (probe transfer)

Leave-one-tuple-out: for each `(layer, level, slot, offset)` fit the probe on *train*-tuple
positions, evaluate on *held-out*-tuple positions. `heldout_acc` near `bayes_acc` = genuine
abstraction; a big `gap = seen − heldout` = the model keyed on surface form. Only levels
with `M > 1` are analyzed.

In [ ]:
res = spelling_generalization(teacher, student, data[:256], offsets=[-1, 0])
summ = summarize(res)
df = pd.DataFrame([
    {'layer': layer, 'level': level, **{k: round(v, 3) for k, v in stats.items()}}
    for (layer, level), stats in sorted(summ.items())
])
display(df)

# Per-layer seen vs held-out spelling accuracy, with the Bayes anchor.
levels_present = sorted(df['level'].unique())
fig, axes = plt.subplots(1, len(levels_present),
                         figsize=(4.2 * len(levels_present), 3.4), squeeze=False)
for ax, lev in zip(axes[0], levels_present):
    sub = df[df['level'] == lev].sort_values('layer')
    x = np.arange(len(sub))
    ax.bar(x - 0.2, sub['seen_acc'], 0.4, label='seen spellings', color=SERIES[0])
    ax.bar(x + 0.2, sub['heldout_acc'], 0.4, label='held-out spellings', color=SERIES[1])
    ax.plot(x, sub['bayes_acc'], 'k--', marker='o', label='Bayes (spelling-invariant)')
    ax.set_xticks(x, [f'L{int(l)}' for l in sub['layer']])
    ax.set_title(f'level {lev}'); ax.set_ylim(0, 1); ax.set_xlabel('layer')
    ax.spines[['top', 'right']].set_visible(False)
axes[0][0].set_ylabel('unit-decode accuracy'); axes[0][0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

## 3. Prediction variability across spellings

Render the same latent (base-id) sequence into `K` spellings and compare predictions.
`cross_spelling_divergence` is the mean pairwise symmetric-KL across spellings at each
position. The **teacher** is the reference: within a unit it legitimately varies (the next
surface token depends on the current spelling), but at a **base-token boundary** the next
unit is fixed by context, so teacher divergence → 0. Any student divergence *there* is
surface leakage — the model failing to abstract.

In [ ]:
B, K = 128, 5
base_ids = torch.randint(0, teacher.base_teacher.dim, (B, seqlen // total))
spellings = render_spellings(teacher, base_ids, k=K)  # (K, B, seqlen, dim)

slp, tlp = [], []
with torch.no_grad():
    for i in range(K):
        s = spellings[i]
        slp.append(F.log_softmax(student(s[:, :-1, :])[0], dim=-1))
        tlp.append(teacher.unroll(s))
# Align student to teacher: predicts tokens burn_in..seqlen-1.
slp = torch.stack(slp)[:, :, burn_in - 1:, :]
tlp = torch.stack(tlp)
div_s = cross_spelling_divergence(slp).mean(0)  # (seqlen - burn_in,)
div_t = cross_spelling_divergence(tlp).mean(0)

pos = np.arange(burn_in, seqlen)  # absolute surface index of the predicted token
boundary = (pos % total == 0)
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(pos, div_t.numpy(), color=SERIES[2], marker='o', label='teacher (reference)')
ax.plot(pos, div_s.numpy(), color=SERIES[1], marker='o', label='student')
for p in pos[boundary]:
    ax.axvline(p, color='0.8', lw=1, zorder=0)
ax.set_xlabel('predicted surface position (grey = base-token boundary)')
ax.set_ylabel('cross-spelling divergence'); ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

print(f'boundary divergence  teacher={div_t[boundary].mean():.4f}  '
      f'student={div_s[boundary].mean():.4f}  '
      f'(student boundary divergence is the surface-leakage signal)')

## Reading the results

* **Gap (seen − heldout)** near 0 with `heldout_acc ≈ bayes_acc` → the layer represents the
  unit *abstractly*; a large gap → it memorized spellings. Comparing across layers/levels
  shows *where* and *at which level of abstraction* invariance emerges.
* **Student boundary divergence** near 0 → predictions are spelling-invariant where they
  should be; well above the teacher's 0 → surface leakage.

The tiny DEMO student (400 steps) overfits spellings on purpose. Train longer / larger, or
load a real checkpoint, and watch both signals move toward abstraction.